# Lab 8.2: Building a RAG Pipeline

**Course:** Advanced Natural Language Processing (NLP702/806)

**Instructor:** Dr. Fajri Koto

---

In Lab 8.1, we learned how BM25 retrieval works. Now we combine retrieval with a **Large Language Model** to build a complete **Retrieval-Augmented Generation (RAG)** pipeline.

### What We'll Build:

1. **Document chunking** - Split documents into retrievable passages
2. **BM25 Retriever** - Find relevant passages for a question
3. **Prompt construction** - Format retrieved context for the LLM
4. **LLM generation** - Generate answers grounded in retrieved evidence
5. **End-to-end RAG pipeline** - Tie everything together

In [1]:
import json
import textwrap
from collections import Counter

import numpy as np
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from tqdm import tqdm

## Section 1: Document Chunking

Real documents are often too long to use as-is. We need to split them into smaller **chunks** that:
- Fit within the LLM's context window
- Contain focused, coherent information
- Overlap slightly to avoid losing information at boundaries

In [2]:
def chunk_text(text, chunk_size=200, overlap=50):
    """
    Split text into overlapping chunks by word count.
    
    Args:
        text: Input text to chunk
        chunk_size: Target number of words per chunk
        overlap: Number of overlapping words between consecutive chunks
    
    Returns:
        List of text chunks
    """
    words = text.split()
    chunks = []
    start = 0
    
    while start < len(words):
        end = start + chunk_size
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    
    return chunks

# Demonstrate chunking
sample_text = """Retrieval-Augmented Generation combines the strengths of retrieval systems 
and generative language models. The retrieval component searches through a large corpus of 
documents to find passages that are relevant to the input query. These retrieved passages 
are then provided as additional context to the language model, which generates a response 
that is grounded in the retrieved information. This approach helps reduce hallucinations 
and allows the model to access up-to-date information that may not have been present in 
its training data. RAG has become one of the most popular techniques for building reliable 
AI systems that need to answer questions accurately."""

chunks = chunk_text(sample_text, chunk_size=30, overlap=10)
print(f"Original text: {len(sample_text.split())} words")
print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i} ({len(chunk.split())} words):")
    print(f"  {chunk}")

Original text: 99 words
Number of chunks: 5

Chunk 0 (30 words):
  Retrieval-Augmented Generation combines the strengths of retrieval systems and generative language models. The retrieval component searches through a large corpus of documents to find passages that are relevant to the

Chunk 1 (30 words):
  of documents to find passages that are relevant to the input query. These retrieved passages are then provided as additional context to the language model, which generates a response that

Chunk 2 (30 words):
  context to the language model, which generates a response that is grounded in the retrieved information. This approach helps reduce hallucinations and allows the model to access up-to-date information that

Chunk 3 (30 words):
  hallucinations and allows the model to access up-to-date information that may not have been present in its training data. RAG has become one of the most popular techniques for building

Chunk 4 (19 words):
  has become one of the most popular technique

## Section 2: Building the Retriever

Let's set up a BM25 retriever on a real knowledge base.

In [3]:
# Load a Wikipedia-based QA dataset
corpus_dataset = load_dataset("rag-datasets/rag-mini-wikipedia", "text-corpus", split="passages")
qa_dataset = load_dataset("rag-datasets/rag-mini-wikipedia", "question-answer", split="test")

passages = [item["passage"] for item in corpus_dataset]

print(f"Knowledge base: {len(passages)} passages")
print(f"QA pairs: {len(qa_dataset)} questions")
print(f"\nSample QA pair:")
print(f"  Q: {qa_dataset[0]['question']}")
print(f"  A: {qa_dataset[0]['answer']}")

data/test.parquet/part.0.parquet:   0%|          | 0.00/54.4k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/918 [00:00<?, ? examples/s]

Knowledge base: 3200 passages
QA pairs: 918 questions

Sample QA pair:
  Q: Was Abraham Lincoln the sixteenth President of the United States?
  A: yes


In [4]:
class BM25Retriever:
    """BM25-based passage retriever."""
    
    def __init__(self, passages):
        self.passages = passages
        self.tokenized = [p.lower().split() for p in tqdm(passages, desc="Tokenizing")]
        self.index = BM25Okapi(self.tokenized)
        print(f"Indexed {len(passages)} passages")
    
    def retrieve(self, query, top_k=5):
        """Retrieve top-k passages for a query."""
        query_tokens = query.lower().split()
        scores = self.index.get_scores(query_tokens)
        top_indices = np.argsort(scores)[::-1][:top_k]
        
        results = []
        for idx in top_indices:
            results.append({
                "passage": self.passages[idx],
                "score": float(scores[idx]),
                "index": int(idx)
            })
        return results

retriever = BM25Retriever(passages)

Tokenizing: 100%|██████████| 3200/3200 [00:00<00:00, 238312.73it/s]

Indexed 3200 passages


In [5]:
# Test retrieval quality
# Note: This dataset has yes/no gold answers, so we check if key question terms
# appear in retrieved passages (since literal "yes"/"no" won't be in the text)
sample_questions = [qa_dataset[i]["question"] for i in range(5)]
sample_answers = [qa_dataset[i]["answer"] for i in range(5)]

for q, a in zip(sample_questions, sample_answers):
    results = retriever.retrieve(q, top_k=3)
    print(f"\nQ: {q}")
    print(f"Gold answer: {a}")
    print(f"Top retrieved passage (score={results[0]['score']:.3f}):")
    print(f"  {results[0]['passage'][:200]}...")
    
    # Check if answer appears in top-k passages
    # For yes/no answers, check if key terms from the question appear instead
    if a.lower().strip() in ["yes", "no"]:
        # Extract key terms from question (skip stopwords)
        stopwords = {"was", "is", "the", "a", "an", "did", "do", "does", "his", "her", "of", "in", "on", "to", "and", "or"}
        key_terms = [w.lower().strip("?.,!") for w in q.split() if w.lower().strip("?.,!") not in stopwords and len(w) > 2]
        terms_found = sum(1 for t in key_terms if any(t in r["passage"].lower() for r in results))
        coverage = terms_found / max(len(key_terms), 1)
        print(f"  Relevant passage found (term coverage): {coverage:.0%} ({terms_found}/{len(key_terms)} key terms)")
    else:
        answer_found = any(a.lower() in r["passage"].lower() for r in results)
        print(f"  Answer in top-3: {'Yes' if answer_found else 'No'}")


Q: Was Abraham Lincoln the sixteenth President of the United States?
Gold answer: yes
Top retrieved passage (score=27.376):
  Abraham Lincoln (February 12, 1809 â April 15, 1865) was the sixteenth President of the United States, serving from March 4, 1861 until his assassination. As an outspoken opponent of the expansion o...
  Relevant passage found (term coverage): 100% (6/6 key terms)

Q: Did Lincoln sign the National Banking Act of 1863?
Gold answer: yes
Top retrieved passage (score=21.031):
  Lincoln did not show the pledge to his cabinet, but asked them to sign the sealed envelope....
  Relevant passage found (term coverage): 100% (6/6 key terms)

Q: Did his mother die of pneumonia?
Gold answer: no
Top retrieved passage (score=11.720):
  Alice Hathaway Lee Roosevelt (July 29, 1861 in Chestnut Hill, Massachusetts â February 14 1884 in Manhattan, New York) was the first wife of Theodore Roosevelt and mother of their only child togethe...
  Relevant passage found (term coverage

## Section 3: Prompt Construction

The prompt template is crucial for RAG. We need to instruct the LLM to:
1. Use the provided context to answer the question
2. Not make up information beyond what's in the context
3. Indicate when the context is insufficient

In [6]:
def build_rag_prompt(query, retrieved_passages, max_context_words=500):
    """
    Build a RAG prompt from query and retrieved passages.
    
    Args:
        query: The user's question
        retrieved_passages: List of retrieved passage dicts
        max_context_words: Maximum total words for context
    
    Returns:
        Formatted prompt string
    """
    # Build context from retrieved passages (respecting token budget)
    context_parts = []
    word_count = 0
    
    for i, passage in enumerate(retrieved_passages):
        passage_words = len(passage["passage"].split())
        if word_count + passage_words > max_context_words:
            break
        context_parts.append(f"[Passage {i+1}] {passage['passage']}")
        word_count += passage_words
    
    context = "\n\n".join(context_parts)
    
    prompt = f"""Answer the question based ONLY on the provided context. If the context does not contain enough information to answer, say "I cannot answer this based on the provided context."

Context:
{context}

Question: {query}

Answer:"""
    
    return prompt

# Demonstrate prompt construction
query = "What is the capital of France?"
results = retriever.retrieve(query, top_k=3)
prompt = build_rag_prompt(query, results)

print("Generated RAG Prompt:")
print("=" * 60)
print(prompt)
print("=" * 60)
print(f"\nPrompt length: {len(prompt.split())} words")

Generated RAG Prompt:
Answer the question based ONLY on the provided context. If the context does not contain enough information to answer, say "I cannot answer this based on the provided context."

Context:
[Passage 1] Montevideo, capital of the country. A view of pedestrian street in the Ciudad Vieja, former Spanish citadel

[Passage 2] Romania has been a member of the European Union since January 1 2007, and has the ninth largest territory in the EU and with 22 million people      it has the 7th largest population among the EU member states. Its capital and largest city is Bucharest (   ), the sixth largest city in the EU with almost 2.5 million people. In 2007, Sibiu, a large city in Transylvania, was chosen as European Capital of Culture.    Romania joined NATO on March 29, 2004, and is also a member of the Latin Union, of the Francophonie and of OSCE.

[Passage 3] Jakarta, the capital of Indonesia and its largest commercial center

Question: What is the capital of France?

Answer

## Section 4: LLM Generation

Now let's connect the retriever to an LLM. We'll use the OpenAI-compatible API format, which works with many providers (OpenAI, Together AI, local models via vLLM/Ollama, etc.).

**Setup:** Set your API key and base URL below. If using a local model, point `base_url` to your local server.

### Setting Up Ollama

In this lab, we use [Ollama](https://ollama.com/) to run an LLM locally. Follow these steps before proceeding:

1. **Install Ollama:**
   - **macOS:** `brew install ollama` or download from [ollama.com/download](https://ollama.com/download)
   - **Linux:** `curl -fsSL https://ollama.com/install.sh | sh`
   - **Windows:** Download the installer from [ollama.com/download](https://ollama.com/download)

2. **Start the Ollama server:**
   ```bash
   ollama serve
   ```
   > This runs in the background on `http://localhost:11434`. Keep this terminal open.

3. **Pull the model** (in a separate terminal):
   ```bash
   ollama pull llama3.2:3b
   ```
   > This downloads ~2 GB. You only need to do this once.

4. **Verify it's working:**
   ```bash
   ollama list          # Should show llama3.2:3b
   curl http://localhost:11434/v1/models   # Should return a JSON response
   ```

In [7]:
from openai import OpenAI

# Configure your LLM endpoint
# Option 1: OpenAI
# client = OpenAI(api_key="your-key-here")
# model_name = "gpt-4o-mini"

# Option 2: Together AI (or any OpenAI-compatible API)
# client = OpenAI(base_url="https://api.together.xyz/v1", api_key="your-key-here")
# model_name = "meta-llama/Llama-3.1-8B-Instruct-Turbo"

# Option 3: Local model via Ollama
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
model_name = "llama3.2:3b"

print(f"Using model: {model_name}")

Using model: llama3.2:3b


In [8]:
def generate_answer(prompt, model=model_name, max_tokens=256, temperature=0.0):
    """
    Generate an answer using the LLM.
    """
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful assistant that answers questions based on the provided context. Be concise and accurate."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_tokens,
        temperature=temperature
    )
    return response.choices[0].message.content

# Test generation without RAG (closed-book)
query = "What is the capital of France?"
closed_book_answer = generate_answer(f"Answer this question concisely: {query}")
print(f"Closed-book answer: {closed_book_answer}")

APIConnectionError: Connection error.

## Section 5: End-to-End RAG Pipeline

In [ ]:
class RAGPipeline:
    """
    End-to-end Retrieval-Augmented Generation pipeline.
    """
    
    def __init__(self, retriever, llm_client, model_name, top_k=3):
        self.retriever = retriever
        self.client = llm_client
        self.model_name = model_name
        self.top_k = top_k
    
    def answer(self, query, verbose=False):
        """
        Answer a question using RAG.
        
        Returns:
            dict with 'answer', 'retrieved_passages', and 'prompt'
        """
        # Step 1: Retrieve
        retrieved = self.retriever.retrieve(query, top_k=self.top_k)
        
        if verbose:
            print(f"Retrieved {len(retrieved)} passages")
            for i, r in enumerate(retrieved):
                print(f"  [{i+1}] (score={r['score']:.3f}) {r['passage'][:80]}...")
        
        # Step 2: Build prompt
        prompt = build_rag_prompt(query, retrieved)
        
        # Step 3: Generate
        answer = generate_answer(prompt, model=self.model_name)
        
        return {
            "query": query,
            "answer": answer,
            "retrieved_passages": retrieved,
            "prompt": prompt
        }

# Create the RAG pipeline
rag = RAGPipeline(retriever, client, model_name, top_k=3)
print("RAG pipeline ready!")

RAG pipeline ready!


In [ ]:
# Test the full pipeline
test_questions = [
    "What is the capital of France?",
    "Who wrote Romeo and Juliet?",
    "What is the speed of light?",
    "When was the United Nations founded?",
    "What causes the seasons on Earth?"
]

for question in test_questions:
    result = rag.answer(question, verbose=True)
    print(f"\nQ: {result['query']}")
    print(f"A: {result['answer']}")
    print("-" * 60)

Retrieved 3 passages
  [1] (score=13.665) Montevideo, capital of the country. A view of pedestrian street in the Ciudad Vi...
  [2] (score=13.650) Romania has been a member of the European Union since January 1 2007, and has th...
  [3] (score=13.304) Jakarta, the capital of Indonesia and its largest commercial center...

Q: What is the capital of France?
A: I cannot answer this based on the provided context. The context only mentions the capitals of Montevideo (Uruguay), Bucharest (Romania), and Jakarta (Indonesia), but does not mention the capital of France.
------------------------------------------------------------
Retrieved 3 passages
  [1] (score=10.164) Tesla was prone to alienating himself and was generally soft-spoken. However, wh...
  [2] (score=9.350) * Aristotle wrote how everything moved, and must be moved by something....
  [3] (score=9.147) Blaise Pascal ( ), (June 19 1623   August 19 1662) was a French mathematician, p...

Q: Who wrote Romeo and Juliet?
A: I cannot ans

## Section 6: RAG vs Closed-Book Comparison

Let's systematically compare RAG answers with closed-book (no retrieval) answers.

In [ ]:
# Compare RAG vs closed-book on a subset of the QA dataset
n_eval = 10
eval_questions = [qa_dataset[i]["question"] for i in range(n_eval)]
eval_answers = [qa_dataset[i]["answer"] for i in range(n_eval)]

results_comparison = []

for q, gold in zip(eval_questions, eval_answers):
    # RAG answer
    rag_result = rag.answer(q)
    
    # Closed-book answer
    closed_prompt = f"Answer this question concisely in one sentence: {q}"
    closed_answer = generate_answer(closed_prompt)
    
    results_comparison.append({
        "question": q,
        "gold_answer": gold,
        "rag_answer": rag_result["answer"],
        "closed_book_answer": closed_answer
    })
    
    print(f"Q: {q}")
    print(f"  Gold:        {gold}")
    print(f"  RAG:         {rag_result['answer'][:150]}")
    print(f"  Closed-book: {closed_answer[:150]}")
    print()

Q: Was Abraham Lincoln the sixteenth President of the United States?
  Gold:        yes
  RAG:         Yes, according to Passage 1, Abraham Lincoln was indeed the sixteenth President of the United States.
  Closed-book: Yes, Abraham Lincoln was indeed the 16th President of the United States, serving from March 4, 1861, until his assassination on April 15, 1865.

Q: Did Lincoln sign the National Banking Act of 1863?
  Gold:        yes
  RAG:         Yes, according to Passage 3, Lincoln signed the National Banking Acts of 1863.
  Closed-book: The National Banking Act was actually signed into law by President Abraham Lincoln on February 25, 1863, but it was not a signature from him as presid

Q: Did his mother die of pneumonia?
  Gold:        no
  RAG:         I cannot answer this based on the provided context. The text does not mention pneumonia as a cause of death for Isaac Newton's mother, Hannah Ayscough
  Closed-book: I don't have enough information to answer your question accurately

### Key Observations

**When RAG helps:**
- Questions about specific facts, dates, or numbers
- Questions requiring up-to-date information
- Questions about less common topics where the LLM might hallucinate

**When RAG may not help (or hurt):**
- Simple, well-known facts the LLM already knows
- When retrieved passages are irrelevant (garbage-in, garbage-out)
- When the question requires reasoning beyond what's in the passages

**The retriever quality is the bottleneck.** If BM25 retrieves the wrong passages, the LLM will either ignore them or be misled. This motivates the exercise in Lab 8.4, where you'll explore ways to improve retrieval.